# Drive an MCP server with a real LLM

**Week 10 · Session 2 · Notebook 03**

Yesterday you wrote an agent loop by hand, and every tool schema with it.

Today the tools come from a **server**, discovered at runtime over MCP. The loop does
not change. That is the entire point, and by the end of this notebook you will have
counted the changed lines yourself.

```
yesterday:   TOOLS = [SEARCH_TOOL, CALC_TOOL, ...]      # you typed this
             out   = DISPATCH[name](**args)             # local function

today:       TOOLS = to_openai(await session.list_tools())
             out   = await session.call_tool(name, args)
```

**Before you start:** seed the database once.

```bash
python3 mcp_server/seed_data.py
```

In [ ]:
%pip install -q openai mcp python-dotenv

In [ ]:
import asyncio, json, os, sys
from pathlib import Path
from openai import OpenAI

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4.1-mini"

# Find the server relative to this notebook, whether you opened it from
# week10/ or week10/notebooks/.
SERVER = next(p for p in [Path("mcp_server/incident_server.py"),
                          Path("../mcp_server/incident_server.py")] if p.exists())
SERVER = SERVER.resolve()
print("server:", SERVER)

> **Note on `asyncio` in notebooks.** The MCP client SDK is async. Jupyter already runs
> an event loop, so `await` works directly in a cell — you do **not** call
> `asyncio.run()` here. In a plain `.py` script you would (see `client_demo.py`).

---
## Step 1 — Connect and look around

Three things happen on connect: `initialize` negotiates the protocol version, then
`tools/list` asks what is available. Nothing is hard-coded.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def inspect_server():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            info = await session.initialize()
            print(f"connected to: {info.serverInfo.name} v{info.serverInfo.version}\n")

            tools = (await session.list_tools()).tools
            print(f"TOOLS ({len(tools)})")
            for t in tools:
                req = ", ".join(t.inputSchema.get("required", []))
                print(f"  {t.name}({req})")
                print(f"      {t.description.splitlines()[0]}")

            res = (await session.list_resources()).resources
            tmpl = (await session.list_resource_templates()).resourceTemplates
            print(f"\nRESOURCES ({len(res)} static, {len(tmpl)} templated)")
            for x in res:
                print(f"  {x.uri}")
            for x in tmpl:
                print(f"  {x.uriTemplate}")

            prompts = (await session.list_prompts()).prompts
            print(f"\nPROMPTS ({len(prompts)})")
            for p in prompts:
                args = ", ".join(a.name for a in (p.arguments or []))
                print(f"  /{p.name}({args})")


await inspect_server()

Nothing about those six tools was written in this notebook. They were **discovered** —
the server described itself, over the protocol, at runtime.

Add a seventh tool to the server, restart, and this cell prints seven. No client code
changes. That is the M+N property from the slides, made concrete.

---
## Step 2 — Call a tool directly, no model involved

Debug the server before adding a model. A broken server plus a model is two problems.

In [ ]:
async def call(name, args):
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, args)
            return "\n".join(c.text for c in result.content
                              if getattr(c, "text", None))


print(await call("service_health", {"service": "checkout-api"}))
print()
print(await call("search_incidents", {"query": "connection pool", "limit": 2}))

In [ ]:
# Now deliberately bad input. A good MCP tool returns a plain-English error as a
# RESULT -- not an exception. The model reads that string and can recover from it.
print(await call("create_ticket", {"title": "x", "severity": "SEV9",
                                   "service": "checkout-api", "body": "y"}))
print(await call("get_incident", {"incident_id": "INC-9999"}))
print(await call("service_health", {"service": "does-not-exist"}))

Read those three messages as if you were the model. Each one tells you **what was
wrong** and **what to do next** — "Known services: ...", "Use search_incidents to find
one". That is what makes an agent able to recover instead of looping.

This is yesterday's *"the agent reads your error message"* lesson, now enforced at the
protocol level.

---
## Step 3 — The bridge: MCP schemas → OpenAI schemas

Here is the whole adapter. MCP gives you a name, a description and a JSON Schema.
OpenAI wants a name, a description and a JSON Schema.

That is not a coincidence — it is the point of standardising on JSON Schema.

In [ ]:
def to_openai(mcp_tools):
    """Convert MCP tool definitions into OpenAI function-calling schemas."""
    return [{
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": t.inputSchema or {"type": "object", "properties": {}},
        },
    } for t in mcp_tools]


async def show_bridge():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai((await session.list_tools()).tools)
            print(f"{len(tools)} OpenAI-ready schemas\n")
            print(json.dumps(tools[0], indent=2)[:700])


await show_bridge()

Notice the `description` — it is the **docstring** from `incident_server.py`, shipped
verbatim to the model. Yesterday's rule ("tool descriptions are prompts") now has a
mechanical consequence: a lazy docstring becomes a lazy prompt in *every* app that
connects to your server.

---
## Step 4 — The loop

Compare this against `run_agent` in notebook 01. Two lines differ.

In [ ]:
SYSTEM = """You are an incident response assistant for an engineering team.

TOOL POLICY
- Search past incidents before proposing any fix. Someone has usually seen it before.
- Check the runbook before recommending an action, and say which one you used.
- Use list_services if you are unsure of an exact service name.
- Never create a ticket unless the user explicitly asks for one.

STOPPING CONDITION
- Stop once you can answer. Do not keep searching for more context.

OUTPUT
- A short recommendation, then the incident ids and runbook sections you relied on.

FAILURE
- If a tool returns an error twice, stop and report what you tried. Never invent an
  incident id, a service name, or a runbook step."""

MAX_STEPS = 8


async def run_mcp_agent(goal, verbose=True):
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ---------- CHANGE 1: the tool list is fetched, not typed ----------
            tools = to_openai((await session.list_tools()).tools)

            messages = [{"role": "system", "content": SYSTEM},
                        {"role": "user", "content": goal}]

            for step in range(1, MAX_STEPS + 1):
                reply = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=tools,
                ).choices[0].message
                messages.append(reply)

                if not reply.tool_calls:
                    return reply.content

                if verbose and reply.content:
                    print(f"  [{step}] THOUGHT     {reply.content.strip()[:100]}")

                for c in reply.tool_calls:
                    args = json.loads(c.function.arguments or "{}")
                    if verbose:
                        print(f"  [{step}] ACTION      {c.function.name}({args})")

                    # ------ CHANGE 2: dispatch goes over the protocol ------
                    try:
                        res = await session.call_tool(c.function.name, args)
                        out = "\n".join(x.text for x in res.content
                                        if getattr(x, "text", None))
                    except Exception as exc:
                        out = f"ERROR: {type(exc).__name__}: {exc}"

                    if verbose:
                        first = out.splitlines()[0] if out.strip() else "(empty)"
                        print(f"      OBSERVATION {first[:84]}")
                    messages.append({"role": "tool", "tool_call_id": c.id,
                                     "content": out[:4000]})

            return "Stopped: hit the step budget."

In [ ]:
answer = await run_mcp_agent(
    "checkout-api p99 latency just jumped to 4 seconds. Has this happened before, "
    "and what should I do first?"
)
print("\n" + answer)

**Count the differences from notebook 01:**

| | Notebook 01 | Notebook 03 |
|---|---|---|
| tool list | a Python list you typed | `session.list_tools()` |
| dispatch | `DISPATCH[name](**args)` | `session.call_tool(name, args)` |
| the `while` loop | — | *identical* |
| message handling | — | *identical* |
| stop condition | — | *identical* |
| guardrails | — | *identical* |

Two lines. In exchange, that server also works in Claude Desktop, in Cursor, and in
anything anyone writes next year.

---
## Step 5 — Watch it plan

Give it a question that needs several tools and no obvious order.

In [ ]:
print(await run_mcp_agent("Which service has the most open incidents, and who is "
                          "on call for it?"))

Look at the trace: `list_services` first, then **several `service_health` calls in the
same step**. Those run in parallel — the model requested them all at once, and the
`for c in reply.tool_calls` loop handled it without any special code.

Parallel tool calls were always available in the loop you wrote yesterday. You just
never asked a question that needed them.

---
## Step 6 — The write, and the human gate

`create_ticket` is the only tool that changes anything. Yesterday's rule applies with
more force here, because the server has no idea who is calling it.

In [ ]:
async def run_gated(goal, verbose=True):
    """Same loop, with a human gate in front of anything that writes."""
    WRITES = {"create_ticket"}

    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai((await session.list_tools()).tools)
            messages = [{"role": "system", "content": SYSTEM},
                        {"role": "user", "content": goal}]

            for step in range(1, MAX_STEPS + 1):
                reply = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=tools,
                ).choices[0].message
                messages.append(reply)
                if not reply.tool_calls:
                    return reply.content

                for c in reply.tool_calls:
                    args = json.loads(c.function.arguments or "{}")

                    if c.function.name in WRITES:
                        print("\n" + "=" * 62)
                        print(f"  APPROVAL REQUIRED — {c.function.name}")
                        print("=" * 62)
                        for k, v in args.items():
                            print(f"  {k:<10}: {str(v)[:200]}")
                        print("=" * 62)
                        if input("  Approve? [y/N] ").strip().lower() != "y":
                            out = ("DENIED by the human operator. The ticket was NOT "
                                   "created. Do not retry; report this to the user.")
                            messages.append({"role": "tool", "tool_call_id": c.id,
                                             "content": out})
                            print("  -> denied\n")
                            continue

                    if verbose:
                        print(f"  [{step}] {c.function.name}({args})")
                    res = await session.call_tool(c.function.name, args)
                    out = "\n".join(x.text for x in res.content
                                    if getattr(x, "text", None))
                    messages.append({"role": "tool", "tool_call_id": c.id,
                                     "content": out[:4000]})

            return "Stopped: hit the step budget."

In [ ]:
# Answer 'n' the first time. A good system prompt makes it report the refusal
# rather than retrying forever.
print(await run_gated(
    "Look up INC-1000, then create a SEV2 ticket on checkout-api titled "
    "'Add connection-pool leak check to CI' summarising the follow-up."
))

The gate lives in the **host**, not the server — the server has no idea who is asking,
so it cannot make that judgement. That is a direct consequence of the architecture:
*the server never sees the model.*

This is the same design as the ladder from yesterday. MCP moved where the tool lives.
It did not move where the responsibility lives.

---
## Step 7 — Resources and prompts

Tools are model-controlled. The other two primitives are not, so they do not go into
`tools=[...]` at all — the **host** decides when to use them.

In [ ]:
async def use_resource_and_prompt():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # RESOURCE -- the app chooses to load this into context.
            doc = await session.read_resource("runbook://checkout-api")
            text = doc.contents[0].text
            print("RESOURCE runbook://checkout-api")
            print("\n".join(text.splitlines()[:8]), "\n...\n")

            # PROMPT -- the user picks this from a menu; it returns a workflow.
            p = await session.get_prompt("triage", {"symptom": "checkout 500s"})
            print("PROMPT /triage")
            print(p.messages[0].content.text[:420])


await use_resource_and_prompt()

In [ ]:
# A resource loaded straight into the prompt -- no tool call, no model decision.
# Use this when you KNOW the document is needed and don't want to pay for a
# round trip to find that out.
async def ask_with_runbook(question, service):
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            doc = await session.read_resource(f"runbook://{service}")
            runbook = doc.contents[0].text

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content":
                   f"Using ONLY this runbook, answer concisely.\n\n"
                   f"RUNBOOK:\n{runbook}\n\nQUESTION: {question}"}],
    )
    return resp.choices[0].message.content


print(await ask_with_runbook("What is the very first thing to check on a p99 spike?",
                             "checkout-api"))

Notice the difference in who decided:

- **Tool** — the model decided it needed the runbook and asked for it.
- **Resource** — *you* decided, in code, before the model saw anything.

Resources cost one fewer round trip and remove a decision the model could get wrong.
Use them when you already know what context is needed.

---
## What you built

```python
to_openai()        # 10 lines. MCP tool schemas -> OpenAI tool schemas.
run_mcp_agent()    # notebook 01's loop, tools discovered at runtime
run_gated()        # the same, with a human gate on writes
```

The server is a separate process that knows nothing about OpenAI, this notebook, or
you. Point Claude Desktop at it and it works there too, unchanged.

---
## Exercises

**1. Add a tool to the server** — `incidents_by_tag(tag)` — restart, and re-run
step 1. The client picks it up with no changes. That is the whole value proposition in
one exercise.

**2. Break a description.** Replace `search_incidents`'s whole docstring with the
single vague line `Searches things.`, restart, and re-run step 5. Count the tool calls
before and after.

**3. Swap the model.** Set `MODEL = "gpt-4.1"` and compare the trajectories on step 5.
Does the stronger model use fewer steps?

**4. Two servers at once.** Run a second MCP server (try the official `filesystem` one)
and merge both tool lists into `tools=[...]`. Namespacing collisions are a real problem
— how would you solve it?

**5. Measure it.** Import `evaluate()` from notebook 02 and point it at
`run_mcp_agent`. Write five incident-triage tasks with expected tool sequences. You now
have a regression test for an MCP server.